# Read SDTMIG — the base graph of the SDTMIG v3.4 variable table

The pinned SDTMIG v3.4 variable table (CSV, cdisc.org sign-in), as published. Sign-in content: the graph, the view and Provenance.json are written locally and not published (`.gitignore`).

1. pin check per file (`Reference_Versions.json`);
2. each file as a text table (`cfa.read.read_csv_table`, RFC 4180); the header must equal the schema's columns;
3. one graph for the source: each table by its CSVW table description, generated from `schema/sdtmig.yaml` (`cfa.rule.table_to_graph`, CSVW minimal mode);
4. structural SHACL (generated by gen-shacl): must conform;
5. writes `sdtmig.ttl` (longturtle) and its graph hash;
6. the Excel view `SDTMIG.xlsx`, rebuilt from the graph as written; each view's text hash must equal its source table's text hash;
7. `Provenance.json`.

The rdf-tabular cross-check of the same files is in `checks/CSVW_Crosscheck.ipynb`.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

import pyshacl
from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph

from cfa import SCHEMA_DIR
from cfa.graph import graph_sha256, write_longturtle
from cfa.pins import checked_source_file, table_file
from cfa.provenance import flat, provenance, write_provenance
from cfa.read import read_csv_table, read_xlsx_table
from cfa.rule import (check_module, csvw_metadata, graph_iri, key_columns, load_module, module_source,
                      pct, table_to_graph)
from cfa.view import schema_sheet, text_sha256, view_frame, write_view

MODULE = SCHEMA_DIR / 'sdtmig.yaml'
OUT = REPO_ROOT / 'base' / 'sdtmig'

sv = load_module(MODULE)
check_module(sv)
SOURCE = module_source(sv)
TABLES = list(sv.all_classes())
print(SOURCE, TABLES)

## The generated contract is current

The CSVW table group and the SHACL shapes in `schema/` (written by `checks/00_Schema.ipynb`) must equal what the module generates now.

In [ ]:
CSVW_PATH = SCHEMA_DIR / 'csvw' / f'{SOURCE}.csv-metadata.json'
SHACL_PATH = SCHEMA_DIR / 'shacl' / f'{SOURCE}.shacl.ttl'
metas = {t: csvw_metadata(sv, t, f'{pct(t)}.csv') for t in TABLES}
group = json.loads(CSVW_PATH.read_text(encoding='utf-8'))
described = group['tables'] if 'tables' in group else [group]
assert [{k: v for k, v in d.items() if k != '@context'} for d in described] == [{k: v for k, v in metas[t].items() if k != '@context'} for t in TABLES], 'CSVW not current: run checks/00_Schema'
assert SHACL_PATH.read_text(encoding='utf-8') == ShaclGenerator(str(MODULE)).serialize(), 'SHACL not current: run checks/00_Schema'
print('contract current')

## 1–3. Pin check, the tables as text, the graph

In [ ]:
files, tables = {}, {}
g = Graph()
for t in TABLES:
    files[t] = checked_source_file(SOURCE, table_file(SOURCE, t))
    tables[t] = read_csv_table(files[t])
    before = len(g)
    g += table_to_graph(metas[t], tables[t])
    print(f'{t}: rows {len(tables[t])}, columns {len(tables[t].columns)}, triples {len(g) - before}')

## 4. Structural SHACL

In [ ]:
conforms, _, report = pyshacl.validate(g, shacl_graph=Graph().parse(SHACL_PATH))
assert conforms, report
print('triples', len(g), '| SHACL conforms')

## 5. Write the graph, its hash

In [ ]:
TTL = OUT / f'{SOURCE}.ttl'
write_longturtle(g, TTL, {'ig': f'https://kerfors.github.io/cdisc-for-all/def/{SOURCE}/{pct(TABLES[0])}#'})
written = Graph().parse(TTL)
GRAPH_HASH = graph_sha256(written)
assert GRAPH_HASH == graph_sha256(g)
print(TTL.name, GRAPH_HASH)

## 6. The Excel views, lossless

In [ ]:
views, SOURCE_HASH, VIEW_HASH = {}, {}, {}
for t in TABLES:
    keys = key_columns(sv, t)
    views[t] = view_frame(written, sv, t)
    SOURCE_HASH[t] = text_sha256(tables[t], keys)
    VIEW_HASH[t] = text_sha256(views[t], keys)
    assert VIEW_HASH[t] == SOURCE_HASH[t], f'{t}: the view differs from the source table'
    print(f'{t}: text hash, source = view: {VIEW_HASH[t]}')

## 7. Provenance; write the views and read them back

In [ ]:
prov = provenance(SOURCE, [files[t] for t in TABLES], [MODULE, CSVW_PATH, SHACL_PATH], graph_iri(SOURCE),
                  len(written), GRAPH_HASH, SOURCE_HASH, VIEW_HASH)
write_provenance(OUT / 'Provenance.json', prov)
for t in TABLES:
    keys = key_columns(sv, t)
    xlsx = OUT / f'{t}.xlsx'
    readme = [
        f'SDTMIG v3.4 variable table ({files[t].name}), as published. Sign-in content: not for redistribution.',
        f'Sheet SDTMIG: rebuilt from the graph {TTL.name}; columns in published order, rows sorted by the key ({", ".join(keys)}).',
        'Every cell is text. The Schema sheet gives each column\'s property IRI; Provenance gives the pins and the hashes.',
        'cdisc-for-all: https://github.com/kerfors/cdisc-for-all',
    ]
    write_view(xlsx, readme, {'SDTMIG': views[t]}, schema_sheet(sv, [t]), flat(prov))
    assert text_sha256(read_xlsx_table(xlsx, 'SDTMIG'), keys) == SOURCE_HASH[t]
    print(xlsx.name, 'read back: text hash equal')